

## Q2: Add/ Sub/ Multiply each element of vector hA by const k=12, put results in hC 

In [1]:
%%writefile q2.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define N          60
#define LOW        10
#define HIGH       99
#define BLOCK_SIZE  8
#define K          12

// ── CUDA Kernels 
__global__ void numAddKernel(int *dA, int *dC, int dk, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        dC[i] = dA[i] + dk;
}

__global__ void numSubKernel(int *dA, int *dC, int dk, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        dC[i] = dA[i] - dk;
}

__global__ void numMulKernel(int *dA, int *dC, int dk, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        dC[i] = dA[i] * dk;
}

// ── Host Functions 
__host__ void numAdd(int *hA, int *hC, int hk, int n)
{
    int *dA, *dC;
    int size = n * sizeof(int);

    cudaMalloc((void**)&dA, size);
    cudaMalloc((void**)&dC, size);

    cudaMemcpy(dA, hA, size, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);

    numAddKernel<<<DimGrid, DimBlock>>>(dA, dC, hk, n);

    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    cudaFree(dA); cudaFree(dC);
}

__host__ void numSub(int *hA, int *hC, int hk, int n)
{
    int *dA, *dC;
    int size = n * sizeof(int);

    cudaMalloc((void**)&dA, size);
    cudaMalloc((void**)&dC, size);

    cudaMemcpy(dA, hA, size, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);

    numSubKernel<<<DimGrid, DimBlock>>>(dA, dC, hk, n);

    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    cudaFree(dA); cudaFree(dC);
}

__host__ void numMul(int *hA, int *hC, int hk, int n)
{
    int *dA, *dC;
    int size = n * sizeof(int);

    cudaMalloc((void**)&dA, size);
    cudaMalloc((void**)&dC, size);

    cudaMemcpy(dA, hA, size, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, 1, 1);
    dim3 DimGrid((int)ceil((float)n / BLOCK_SIZE), 1, 1);

    numMulKernel<<<DimGrid, DimBlock>>>(dA, dC, hk, n);

    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    cudaFree(dA); cudaFree(dC);
}

// ── Main 
int main()
{
    int *hA, *hC;
    int  hk = K;

    hA = (int*) malloc(N * sizeof(int));
    hC = (int*) malloc(N * sizeof(int));

    srand(time(NULL));
    for(int i=0; i<N; i++)
    {
        hA[i] = rand()%90+10;
    }

    printf("hA: "); for(int i=0;i<N;i++) printf("%4d",hA[i]); printf("\n");

    // Addition
    numAdd(hA, hC, hk, N);
    printf("\nhC = hA + %d:\n", hk);
    for(int i=0;i<N;i++) printf("%4d",hC[i]); printf("\n");

    // Subtraction
    numSub(hA, hC, hk, N);
    printf("\nhC = hA - %d:\n", hk);
    for(int i=0;i<N;i++) printf("%4d",hC[i]); printf("\n");

    // Multiplication
    numMul(hA, hC, hk, N);
    printf("\nhC = hA * %d:\n", hk);
    for(int i=0;i<N;i++) printf("%4d",hC[i]); printf("\n");

    free(hA); free(hC);
    return 0;
}

Writing q2.cu


## Compile and Run:

In [2]:
!nvcc -arch=sm_75 q2.cu -o q2

!./q2

hA:   47  66  44  61  82  15  99  22  37  55  39  49  56  19  87  53  32  48  67  74  71  64  86  75  32  95  92  26  79  96  47  78  62  81  40  45  86  91  19  75  46  48  24  54  20  63  97  94  11  27  68  72  81  55  10  13  12  92  29  44

hC = hA + 12:
  59  78  56  73  94  27 111  34  49  67  51  61  68  31  99  65  44  60  79  86  83  76  98  87  44 107 104  38  91 108  59  90  74  93  52  57  98 103  31  87  58  60  36  66  32  75 109 106  23  39  80  84  93  67  22  25  24 104  41  56

hC = hA - 12:
  35  54  32  49  70   3  87  10  25  43  27  37  44   7  75  41  20  36  55  62  59  52  74  63  20  83  80  14  67  84  35  66  50  69  28  33  74  79   7  63  34  36  12  42   8  51  85  82  -1  15  56  60  69  43  -2   1   0  80  17  32

hC = hA * 12:
 564 792 528 732 984 1801188 264 444 660 468 588 672 2281044 636 384 576 804 888 852 7681032 900 38411401104 312 9481152 564 936 744 972 480 54010321092 228 900 552 576 288 648 240 75611641128 132 324 816 864 972 660 120 156 144